# Customer Churn — Model Building Pipeline

This notebook documents **how the churn model was built** and the **flow from
file to file**. It is a written walkthrough (no code) — each stage below lives
in its own notebook, and each notebook pulls in the previous one with `%run`,
so running the last notebook re-runs the whole chain.


## Flow at a glance

```
data_ingestion.ipynb
        │   (loads raw data from S3, renames a column)
        ▼
preprocess.ipynb            %run data_ingestion.ipynb
        │   (clean → encode → engineer → split → scale → balance)
        ▼
train.ipynb                 %run preprocess.ipynb
        │   (fit 4 candidate models)
        ▼
evaluate.ipynb              %run train.ipynb
        │   (compare models → pick Random Forest)
        ▼
saving_the_model.ipynb      (joblib.dump the chosen model + helpers)
        │
        ▼
upload_to_s3.ipynb          (push the .pkl artifacts back to S3)
```

Because every notebook starts with `%run <previous>.ipynb`, the DataFrame `df`,
the train/test splits and the fitted models all flow forward automatically —
no data is re-loaded or re-computed by hand between stages.


## Step 1 — `data_ingestion.ipynb`  ·  Load the raw data

- Reads the raw dataset straight from S3:
  `s3://customer-churn-02/data/raw/Bank Customer Churn Prediction.csv`
- Shape: **10,000 rows × 12 columns**.
- Renames one column for clarity: **`tenure` → `customer_retention`**.

**Output carried forward:** a pandas DataFrame `df` holding the raw customers.


## Step 2 — `preprocess.ipynb`  ·  Clean, encode, engineer, split, balance

Starts with `%run data_ingestion.ipynb` to inherit `df`, then:

1. **Data quality checks** — confirms there are no duplicate rows and no missing
   values.
2. **Label-encode `gender`** with `LabelEncoder` → **Female = 0, Male = 1**
   (this fitted encoder is saved later so inference encodes gender the same way).
3. **Drop non-predictive columns** — removes **`customer_id`** (an identifier)
   and **`country`**.
4. **Feature engineering** — creates
   **`balance_salary_ratio = balance / (estimated_salary + 1)`**
   (the `+ 1` avoids division-by-zero when salary is 0).
5. **Split** into features `X` and target `churn`, then an **80/20
   stratified** `train_test_split` (stratified so the churn ratio is preserved).
6. **Two feature representations are kept:**
   - `X_train_raw` / `X_test_raw` — **unscaled**, for tree-based models
     (Random Forest, Gradient Boosting, XGBoost) which don't need scaling.
   - `X_train` / `X_test` — **StandardScaler-scaled** numerical columns, for
     Logistic Regression which benefits from scaling.
7. **Handle class imbalance with SMOTE** — churn is ~80/20 imbalanced, so
   **SMOTENC** (SMOTE for mixed numeric + categorical features) resamples the
   **training set only** to a 50/50 balance. SMOTENC is used instead of plain
   SMOTE so the 0/1 categorical columns aren't given fractional values. The test
   set is never resampled, to keep evaluation honest.

**Outputs carried forward:** `X_train`, `X_test`, `y_train`, `y_test` (scaled) and
`X_train_raw`, `X_test_raw`, `y_train_raw` (raw), all SMOTE-balanced on the train side.


## Step 3 — `train.ipynb`  ·  Fit candidate models

Starts with `%run preprocess.ipynb`, then trains **four** models so they can be
compared:

| Model | Trained on |
|-------|------------|
| Logistic Regression | scaled + SMOTE (`X_train`, `y_train`) |
| **Random Forest**   | raw + SMOTE (`X_train_raw`, `y_train_raw`) |
| Gradient Boosting   | raw + SMOTE (`X_train_raw`, `y_train_raw`) |
| XGBoost             | raw + SMOTE (`X_train_raw`, `y_train_raw`) |

The tree-based models are deliberately fit on the **raw (unscaled)** features —
this matters at serving time: **no scaler is needed** to use the final model.


## Step 4 — `evaluate.ipynb`  ·  Compare and choose

Starts with `%run train.ipynb`, then for each model reports a **confusion
matrix**, a **classification report** (precision / recall / F1) and **ROC-AUC**.

Summary of results on the held-out test set:

| Model | Accuracy | Churn recall | ROC-AUC |
|-------|:--------:|:------------:|:-------:|
| Logistic Regression | 0.72 | 0.72 | 0.773 |
| **Random Forest**   | **0.80** | 0.67 | **0.837** |

The **Random Forest** gives the best overall balance (highest ROC-AUC and
accuracy), so it is selected as the model to ship.


## Step 5 — `saving_the_model.ipynb`  ·  Persist the artifacts

The three things needed to reproduce a prediction are saved with `joblib` into
`training/models/`:

| File | What it is | Why it's needed at inference |
|------|-----------|------------------------------|
| `churn_model.pkl` | the trained Random Forest | makes the prediction |
| `feature_columns.pkl` | ordered list of the 10 feature names | features must be in the exact training order |
| `gender_encoder.pkl` | the fitted `LabelEncoder` | encode gender identically (Female→0, Male→1) |

The saved feature order is:
`credit_score, gender, age, customer_retention, balance, products_number,
credit_card, active_member, estimated_salary, balance_salary_ratio`.


## Step 6 — `upload_to_s3.ipynb`  ·  Publish to S3

Uploads the saved `.pkl` artifact(s) from `training/models/` back to the project
bucket, under `s3://customer-churn-02/models/`, so the model can be pulled for
deployment/serving from a central location.

---

### Recap of the file-to-file flow

`data_ingestion → preprocess → train → evaluate → saving_the_model → upload_to_s3`

Each step consumes the previous step's in-memory objects via `%run`, and the
final two steps turn the chosen in-memory model into on-disk and in-S3 artifacts.
